> **中文阅读版**：本文件是 `01_itch_parser.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# NASDAQ TotalView-ITCH: 订单簿数据解析

**第3章：市场微观结构**

**Docker镜像**: `ml4t`

## 目的

本笔记本演示了如何解析纳斯达克的TotalView-ITCH二进制协议。
理解MBO（按订单消息）数据是构建基于市场微观结构机器学习特征的基础。

## 学习目标

完成本笔记本后，您将能够：
- 从二进制文件中读取一个ITCH v5.0消息：找到它的起始位置、长度以及它是二十多种消息类型中的哪一种。
- 使用Python的`struct`模块解包消息的字段，并转换ITCH使用的两种编码 - 从午夜起的纳秒偏移量和以四位隐含小数表示的整数价格。
- 在整个交易日内运行解析，将每种消息类型写入自己的Parquet分区，以便不适用于内存的一天仍然落在磁盘上。
- 加载已解析的一天并检查每种消息类型是否都能正确读取。
- 判断何时Python解析器是合适的工具，何时Rust解析器是合适的。

## 书籍参考

第§3.3节，*从原始消息到限价订单簿* - 大规模二进制解析子节和表3.1. §3.2将ITCH归入其他数据源。

## 关联

- **下游**: `02_itch_lob_reconstruction`（从这些消息构建订单簿）
- **相关**: `09_databento_mbo_analysis`（替代MBO数据源）

## 数据需求

可以使用以下方式下载ITCH样本数据：
bash
python data/equities/market/microstructure/nasdaq_itch_download.py --list   # 列出可用文件
python data/equities/market/microstructure/nasdaq_itch_download.py          # 下载默认样本


文件是从：https://emi.nasdaq.com/ITCH/压缩的，大约5GB。



## ITCH消息类型

[ITCH v5.0规范](https://www.nasdaqtrader.com/content/technicalsupport/specifications/dataproducts/NQTVITCHSpecification.pdf)定义了20多种消息类型：

| 类型 | 名称 | 描述 |
|------|------|-------------|
| **S** | 系统事件 | 市场开盘/收盘事件 |
| **R** | 股票目录 | 交易代码信息和特征 |
| **H** | 交易行为 | 交易暂停、暂停和恢复 |
| **Y** | Reg SHO限制 | 空头交易价格测试限制 |
| **L** | 市场参与者 | 做市商头寸 |
| **V** | MWCB下降水平 | 市场范围熔断器水平 |
| **W** | MWCB状态 | 熔断器突破状态 |
| **A** | 添加订单 | 新的限价订单进入簿 |
| **F** | 添加订单（MPID） | 与A相同，带有市场参与者ID |
| **E** | 订单执行 | 对现有订单的部分/全部执行 |
| **C** | 带价格的订单执行 | 以不同价格执行（隐藏订单） |
| **X** | 订单取消 | 部分取消 |
| **D** | 订单删除 | 完全从簿中删除 |
| **U** | 订单替换 | 修改价格/大小（取消+添加） |
| **P** | 交易 | 非显示执行 |
| **Q** | 交叉交易 | 开盘/收盘交叉 |
| **B** | 破裂交易 | 交易取消 |
| **I** | NOII | 净订单不平衡指标（拍卖） |
| **J** | LULD拍卖领口 | 限价上涨-限价下跌价格带 |
| **K** | IPO报价期 | IPO报价时间 |

通过按时间顺序组合这些消息，我们可以在任何时间点重建订单簿。



In [1]:
"""NASDAQ TotalView-ITCH: Order Book Data Parsing — parse ITCH binary protocol into structured messages."""

import gzip
import os
import shutil
import struct
from collections import Counter, defaultdict
from datetime import date, datetime
from pathlib import Path
from time import time

import polars as pl
from itch_message_specs import (
    FMT_DICT,
    MESSAGE_SPECS,
    NT_DICT,
    flush_to_parquet,
    parse_price4,
    parse_timestamp,
    print_message_formats,
)
from tqdm.auto import tqdm

from data import load_nasdaq_itch
from utils.paths import display_path

In [2]:
SKIP_PARSING = False
# Messages to read before stopping. None parses the whole session, which is what the
# shipped output is built from; a small number is how you try the parser out first.
MAX_MESSAGES = None

解析读取一个目录并写入另一个。`load_nasdaq_itch(get_base_path=True)`
从存储库的数据配置中解析消息根，而不是在这里键入的路径，因此相同的笔记本可以在本地签出和Docker镜像中运行。原始二进制文件位于旁边的`raw/`目录中，这是下载脚本放置它的位置。

`must_exist=False`是使干净启动工作的原因。对于本章中的其他每个笔记本，一个缺失的`messages/`目录是下载指令，加载器有权利拒绝；这个笔记本是创建它的一个，它必须被告知在哪里，在有任何东西之前。该目录本身是在第4节中创建的，解析即将写入：在这里创建它会在没有原始馈送的马机器上留下一个空目录，而一个空目录正是阻止加载器拒绝的原因。



In [3]:
MESSAGE_DIR = load_nasdaq_itch(get_base_path=True, must_exist=False)
ITCH_RAW_DIR = MESSAGE_DIR.parent / "raw"

print(f"Raw ITCH binary (input):  {display_path(ITCH_RAW_DIR)}")
print(f"Parsed messages (output): {display_path(MESSAGE_DIR)}")
raw_present = ITCH_RAW_DIR.exists() and next(ITCH_RAW_DIR.iterdir(), None) is not None
print(f"Raw binary present: {raw_present}")
if not raw_present:
    print(
        "  Fetch it first: "
        "uv run python data/equities/market/microstructure/nasdaq_itch_download.py"
    )


def message_type_dirs() -> list[Path]:
    """Parsed message-type subdirectories, one uppercase letter each.

    Empty before Section 4 has run: `messages/` itself does not exist on a machine that
    has never parsed, so every cell that lists message types goes through here.
    """
    if not MESSAGE_DIR.exists():
        return []
    return [
        d
        for d in sorted(MESSAGE_DIR.iterdir())
        if d.is_dir() and len(d.name) == 1 and d.name.isupper()
    ]

Raw ITCH binary (input):  <ML4T_DATA_PATH>/equities/market/microstructure/nasdaq_itch/raw
Parsed messages (output): <ML4T_DATA_PATH>/equities/market/microstructure/nasdaq_itch/messages
Raw binary present: True


## 1. 消息规范

每个ITCH消息都有一个固定的二进制结构。格式在`itch_message_specs.py`中使用Python的`struct`模块定义。格式代码：
- `H` = 无符号短整型（2字节）
- `I` = 无符号整型（4字节）
- `Q` = 无符号长整型（8字节）
- `s` = 字符（1字节），`Ns` = N个字符
- `>` = 大端字节序



In [4]:
# Show message formats (loaded from utils/itch_message_specs.py)
print_message_formats()

ITCH Message Formats:

  S (System Event             ): 11 bytes - >HH6ss
  R (Stock Directory          ): 38 bytes - >HH6s8sssIss2ssssssIs
  A (Add Order                ): 35 bytes - >HH6sQsI8sI
  F (Add Order MPID           ): 39 bytes - >HH6sQsI8sI4s
  E (Order Executed           ): 30 bytes - >HH6sQIQ
  C (Order Executed with Price): 35 bytes - >HH6sQIQsI
  X (Order Cancel             ): 22 bytes - >HH6sQI
  D (Order Delete             ): 18 bytes - >HH6sQ
  U (Order Replace            ): 34 bytes - >HH6sQQII
  P (Trade                    ): 43 bytes - >HH6sQsI8sIQ
  Q (Cross Trade              ): 39 bytes - >HH6sQ8sIQs
  H (Trading Action           ): 24 bytes - >HH6s8sss4s
  Y (Reg SHO Restriction      ): 19 bytes - >HH6s8ss
  L (Market Participant Position): 25 bytes - >HH6s4s8ssss
  V (MWCB Decline Level       ): 34 bytes - >HH6sQQQ
  W (MWCB Status              ): 11 bytes - >HH6ss
  J (LULD Auction Collar      ): 34 bytes - >HH6s8sIIII
  K (IPO Quoting Period       ): 27 byte

## 2. 二进制解析示例

让我们通过创建和解析一个示例添加订单消息来演示二进制解析的工作原理。

手动打包消息是查看布局的最快方式。添加订单携带的格式为`>HH6sQsI8sI`：大端，两个无符号短整型，一个六字节时间戳，一个八字节订单引用，一个字符的侧面，股票数量，一个八字符填充的股票代码和价格。下面的解包返回的正是这些字段。



手动打包消息是查看布局的最快方式。添加订单携带的格式为`>HH6sQsI8sI`：大端，两个无符号短整型，一个六字节时间戳，一个八字节订单引用，一个字符的侧面，股票数量，一个八字符填充的股票代码和价格。下面的解包返回的正是这些字段。



In [5]:
sample_add_order = struct.pack(
    ">HH6sQsI8sI",
    1234,  # stock_locate
    5678,  # tracking_number
    b"\x00\x00\x00\x00\x00\x01",  # timestamp (1 nanosecond)
    9876543210,  # order_reference_number
    b"B",  # buy_sell_indicator
    100,  # shares
    b"AAPL    ",  # stock (padded to 8 chars)
    1500000,  # price: four implied decimals, so this is $150
)

print(f"Raw Add Order message ({len(sample_add_order)} bytes):")
print(f"  Hex: {sample_add_order.hex()}")

Raw Add Order message (35 bytes):
  Hex: 04d2162e000000000001000000024cb016ea42000000644141504c202020200016e360


In [6]:
# Parse the binary data using our struct format
parsed = struct.unpack(FMT_DICT["A"], sample_add_order)
add_order = NT_DICT["A"]._make(parsed)

print("Parsed Add Order Message:")
print("-" * 40)
for field, value in add_order._asdict().items():
    if isinstance(value, bytes):
        value = value.decode("ascii").strip()
    print(f"  {field:25}: {value}")

Parsed Add Order Message:
----------------------------------------
  stock_locate             : 1234
  tracking_number          : 5678
  timestamp                :      
  order_reference_number   : 9876543210
  buy_sell_indicator       : B
  shares                   : 100
  stock                    : AAPL
  price                    : 1500000


In [7]:
# Apply conversions using helper functions from utils.itch_message_specs
ts_ns = parse_timestamp(add_order.timestamp)
price = parse_price4(add_order.price)

print(f"Timestamp: {ts_ns:,} nanoseconds = {ts_ns / 1e9:.9f} seconds after midnight")
print(f"Price: ${price:.4f}")

Timestamp: 1 nanoseconds = 0.000000001 seconds after midnight
Price: $150.0000


## 3. 当前管道包含的内容

在解析任何内容之前，先进行盘点：原始二进制文件是输入，按消息类型划分的Parquet目录是输出，两者都可以不存在。第一次运行时，解析侧为空，并且保持为空，直到第4节写入内容。解析后的消息在第5节加载，在可以加载内容之后。

In [8]:
# Check what data is available locally
print("ITCH Data Pipeline Status:")
print("-" * 50)

# Step 1: Raw binary from download
raw_files = []
if ITCH_RAW_DIR.exists():
    raw_files = list(ITCH_RAW_DIR.glob("*.gz")) + list(ITCH_RAW_DIR.glob("*.bin"))
print(f"Raw binary files: {len(raw_files)}")
for f in raw_files:
    print(f"  {f.name} ({f.stat().st_size / 1e9:.2f} GB)")

# Step 2: Parsed messages (single uppercase letter = message type)
parsed_types = message_type_dirs()
parsed_with_data = [d for d in parsed_types if list(d.glob("*.parquet"))]
print(f"Parsed message types: {len(parsed_with_data)}")
for msg_dir in parsed_with_data:
    name = MESSAGE_SPECS.get(msg_dir.name, {}).get("name", "Unknown")
    n_files = len(list(msg_dir.glob("*.parquet")))
    print(f"  {msg_dir.name} ({name}): {n_files} files")

ITCH Data Pipeline Status:
--------------------------------------------------
Raw binary files: 2
  01302020.NASDAQ_ITCH50.gz (5.60 GB)
  01302020.NASDAQ_ITCH50.bin (12.95 GB)
Parsed message types: 0


## 4. 完整解析器实现

下面的解析器编写是为了阅读。它使用Python一次处理一条消息，这使得每个步骤都可见，也使得一个完整的交易日大约需要二十分钟。第6节涵盖了Rust解析器，它产生相同的Parquet模式，并且是你会在多天内运行的内容。

### 解析器辅助函数

解析分为三个函数。`read_frame`从文件中取出下一条消息并返回其类型和原始字节；`decode_message`将这些字节转换为Python值的字典；而`parse_itch_file`运行循环，缓冲解码后的消息并以批量形式写入。

In [9]:
def read_frame(f, pbar) -> tuple[str, bytes] | None:
    """Read one ITCH message frame: 2-byte length + 1-byte type + payload.

    Returns (msg_type, payload) on success, or None on EOF/truncation.
    """
    # 2-byte big-endian length prefix (message size including type byte)
    length_bytes = f.read(2)
    if len(length_bytes) < 2:
        return None
    pbar.update(2)

    msg_size = int.from_bytes(length_bytes, "big")

    # 1-byte message type
    msg_type_byte = f.read(1)
    if len(msg_type_byte) < 1:
        print(f"\nWarning: Truncated message at byte {f.tell()}, expected type byte")
        return None
    pbar.update(1)

    msg_type = msg_type_byte.decode("ascii")

    # Payload (msg_size includes type byte, so payload is msg_size - 1)
    payload = f.read(msg_size - 1)
    if len(payload) < msg_size - 1:
        print(f"\nWarning: Truncated payload for message type {msg_type}")
        return None
    pbar.update(msg_size - 1)

    return msg_type, payload

将二进制有效载荷解码为Python字典，将原始时间戳字节转换为纳秒整数，将字节字符串转换为去除ASCII。

In [10]:
def decode_message(msg_type: str, payload: bytes) -> dict | None:
    """Unpack binary payload into a dict, converting timestamps and strings.

    Returns parsed message dict, or None on struct error.
    """
    try:
        parsed = struct.unpack(FMT_DICT[msg_type], payload)
        msg = NT_DICT[msg_type]._make(parsed)._asdict()
    except struct.error:
        return None

    # Convert timestamp: nanoseconds since midnight
    if "timestamp" in msg:
        msg["timestamp"] = int.from_bytes(msg["timestamp"], "big")

    # Decode string fields
    for field, value in msg.items():
        if isinstance(value, bytes):
            msg[field] = value.decode("ascii").strip()

    return msg

主解析器按顺序读取二进制文件，缓冲解码后的消息，并定期刷新到Parquet以限制内存使用。

In [11]:
def parse_itch_file(
    itch_file: Path,
    trading_day: date,
    output_dir: Path,
    max_buffered_messages: int = 10_000_000,
    max_messages: int | None = None,
) -> dict[str, int]:
    """Parse ITCH binary file and store messages as Parquet.

    Args:
        itch_file: Path to binary ITCH file (.bin, not .gz).
        trading_day: Trading date for timestamp construction.
        output_dir: Directory for Parquet output (one subdir per message type).
        max_buffered_messages: Flush threshold (total buffered messages).
        max_messages: Optional limit for testing.

    Returns:
        Dictionary with message type counts.
    """
    # Midnight timestamp for the trading day (ITCH timestamps are nanoseconds offset)
    base_ts = datetime(trading_day.year, trading_day.month, trading_day.day)
    file_counters: dict[str, int] = defaultdict(int)

    buffers = defaultdict(list)
    counts = Counter()
    file_size = itch_file.stat().st_size
    start_time = time()

    with (
        itch_file.open("rb") as f,
        tqdm(total=file_size, desc="Parsing ITCH", unit="B", unit_scale=True) as pbar,
    ):
        while True:
            if max_messages and sum(counts.values()) >= max_messages:
                print(f"\nLimit reached: {max_messages:,} messages")
                break

            frame = read_frame(f, pbar)
            if frame is None:
                break
            msg_type, payload = frame
            counts[msg_type] += 1

            if msg_type not in FMT_DICT:
                continue

            msg = decode_message(msg_type, payload)
            if msg is None:
                continue

            # Check for end of messages
            if msg_type == "S" and msg.get("event_code") == "C":
                print("\nEnd of Messages")
                flush_to_parquet(buffers, output_dir, base_ts, file_counters)
                break

            buffers[msg_type].append(msg)

            # Periodic flush
            if sum(len(v) for v in buffers.values()) >= max_buffered_messages:
                flush_to_parquet(buffers, output_dir, base_ts, file_counters)

    # Final flush
    if any(buffers.values()):
        flush_to_parquet(buffers, output_dir, base_ts, file_counters)

    elapsed = time() - start_time
    total = sum(counts.values())
    print(f"Parsed {total:,} messages in {elapsed:.1f}s ({total / elapsed:,.0f} msg/s)")

    return dict(counts)

In [12]:
# Locate ITCH data file (skip if SKIP_PARSING is set)
if SKIP_PARSING:
    print("SKIP_PARSING=True: skipping ITCH binary parsing (uses pre-parsed data)")
    itch_file = None
    counts = {}
else:
    # Clear any existing parsed data to avoid schema conflicts
    # (Different parser versions may produce different schemas)
    # Set ITCH_KEEP_EXISTING=1 to skip cleanup and use existing data
    clear_existing = os.environ.get("ITCH_KEEP_EXISTING", "0") != "1"
    if clear_existing and MESSAGE_DIR.exists() and list(MESSAGE_DIR.glob("*/part-*.parquet")):
        print(f"Clearing existing parsed data in {MESSAGE_DIR}")
        print("  (Set ITCH_KEEP_EXISTING=1 to keep existing data)")
        shutil.rmtree(MESSAGE_DIR)
        MESSAGE_DIR.mkdir(parents=True, exist_ok=True)

    # Find ITCH file (compressed or uncompressed)
    gz_files = list(ITCH_RAW_DIR.glob("*.gz")) if ITCH_RAW_DIR.exists() else []
    bin_files = list(ITCH_RAW_DIR.glob("*.bin")) if ITCH_RAW_DIR.exists() else []

    if not gz_files and not bin_files:
        raise FileNotFoundError(
            f"No raw ITCH binary found at {ITCH_RAW_DIR}.\n"
            "Download first:\n"
            "  uv run python data/equities/market/microstructure/nasdaq_itch_download.py"
        )

In [13]:
# Decompress if needed and extract trading date
if not SKIP_PARSING and (gz_files or bin_files):
    # Prefer uncompressed, otherwise decompress
    if bin_files:
        itch_file = bin_files[0]
        print(f"Found uncompressed: {itch_file.name}")
    else:
        gz_file = gz_files[0]
        itch_file = gz_file.with_suffix(".bin")

        if not itch_file.exists():
            print(f"Decompressing {gz_file.name}...")
            with gzip.open(gz_file, "rb") as f_in, open(itch_file, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
            print(f"Created: {itch_file.name} ({itch_file.stat().st_size / 1e9:.1f} GB)")
        else:
            print(f"Found: {itch_file.name}")

    # Extract trading date from filename (format: MMDDYYYY.NASDAQ_ITCH50.bin)
    date_str = itch_file.stem.split(".")[0]
    if len(date_str) == 8 and date_str.isdigit():
        trading_day = date(int(date_str[4:8]), int(date_str[:2]), int(date_str[2:4]))
    else:
        trading_day = date(2020, 1, 30)  # Fallback

    print(f"Trading day: {trading_day}")

    # Parse ITCH file (full-day parse: ~22 min on the reference machine)
    if itch_file and itch_file.exists():
        # Created here, not where the path is resolved: an empty messages/ directory
        # left behind by a run that found no raw binary would stop load_nasdaq_itch
        # raising for every later notebook in the chapter.
        MESSAGE_DIR.mkdir(parents=True, exist_ok=True)
        counts = parse_itch_file(
            itch_file=itch_file,
            trading_day=trading_day,
            output_dir=MESSAGE_DIR,
            max_messages=MAX_MESSAGES,
        )

        print("\nMessage counts:")
        for msg_type, count in sorted(counts.items(), key=lambda x: -x[1]):
            name = MESSAGE_SPECS.get(msg_type, {}).get("name", "Unknown")
            print(f"  {msg_type} ({name:25}): {count:>10,}")

Found uncompressed: 01302020.NASDAQ_ITCH50.bin
Trading day: 2020-01-30


Parsing ITCH:   0%|          | 0.00/13.0G [00:00<?, ?B/s]


End of Messages


Parsed 423,285,709 messages in 2158.7s (196,084 msg/s)

Message counts:
  A (Add Order                ): 184,735,355
  D (Order Delete             ): 180,285,101
  U (Order Replace            ): 36,777,372
  E (Order Executed           ):  8,415,610
  X (Order Cancel             ):  4,990,972
  I (NOII                     ):  4,025,192
  F (Add Order MPID           ):  1,875,350
  P (Trade                    ):  1,779,727
  L (Market Participant Position):    216,802
  C (Order Executed with Price):    139,474
  Q (Cross Trade              ):     17,835
  Y (Reg SHO Restriction      ):      9,068
  H (Trading Action           ):      8,921
  R (Stock Directory          ):      8,916
  S (System Event             ):          6
  J (LULD Auction Collar      ):          5
  K (IPO Quoting Period       ):          2
  V (MWCB Decline Level       ):          1


## 5. 消息类型分析

解析为每种消息类型写入了一个Parquet目录，因此现在可以读取存储。下面的单元测试断言在交易消息（`P`）上，因为这是它将要加载的类型。一个完整的会话总是包含它们，所以空的`P`目录意味着解析未完成，而不是这一天恰好没有交易。这种推理只适用于完整解析：设置`MAX_MESSAGES`的运行可以在第一条交易打印之前停止，所以那里报告缺失，其余单元测试与试验实际写入的类型一起工作。

In [14]:
trade_dir = MESSAGE_DIR / "P"
trade_files = sorted(trade_dir.glob("*.parquet")) if trade_dir.exists() else []

if not trade_files and MAX_MESSAGES is not None:
    print(
        f"No trade messages under {display_path(trade_dir)}: the parse stopped after "
        f"{MAX_MESSAGES:,} messages and a `P` need not appear that early. "
        "Set MAX_MESSAGES to None to read the whole session."
    )
else:
    assert trade_files, (
        f"No parsed trade messages at {trade_dir}.\n"
        "Section 4 writes them from the raw binary. If SKIP_PARSING is True, it was skipped\n"
        "and the store has to have been written by an earlier run or by the Rust parser of\n"
        "Section 6. If the raw binary is missing, fetch it first:\n"
        "  uv run python data/equities/market/microstructure/nasdaq_itch_download.py"
    )

    trades = pl.read_parquet(trade_dir / "*.parquet")
    print(f"Loaded {len(trades):,} trade messages")
    print(f"Columns: {trades.columns}")

Loaded 1,779,727 trade messages
Columns: ['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number', 'buy_sell_indicator', 'shares', 'stock', 'price', 'match_number']


In [15]:
# Message type distribution — use lazy scan to count without loading all data
written = [d for d in message_type_dirs() if list(d.glob("*.parquet"))]
print("Parsed Message Types:")
print("-" * 50)
if not written:
    print("  Nothing parsed yet - Section 4 writes these from the raw binary.")
for msg_dir in written:
    count = pl.scan_parquet(msg_dir / "*.parquet").select(pl.len()).collect().item()
    name = MESSAGE_SPECS.get(msg_dir.name, {}).get("name", "Unknown")
    print(f"  {msg_dir.name} ({name:25}): {count:>12,} messages")

Parsed Message Types:
--------------------------------------------------
  A (Add Order                ):  184,735,355 messages
  C (Order Executed with Price):      139,474 messages
  D (Order Delete             ):  180,285,101 messages
  E (Order Executed           ):    8,415,610 messages
  F (Add Order MPID           ):    1,875,350 messages
  H (Trading Action           ):        8,921 messages


  I (NOII                     ):    4,025,192 messages
  J (LULD Auction Collar      ):            5 messages
  K (IPO Quoting Period       ):            2 messages
  L (Market Participant Position):      216,802 messages
  P (Trade                    ):    1,779,727 messages
  Q (Cross Trade              ):       17,835 messages
  R (Stock Directory          ):        8,916 messages
  S (System Event             ):            5 messages
  U (Order Replace            ):   36,777,372 messages
  V (MWCB Decline Level       ):            1 messages
  X (Order Cancel             ):    4,990,972 messages
  Y (Reg SHO Restriction      ):        9,068 messages


In [16]:
# Schema compatibility check — verify we can read each message type
written = [d for d in message_type_dirs() if list(d.glob("*.parquet"))]
print("Schema Compatibility Check:")
print("-" * 50)
if not written:
    print("  Nothing parsed yet - Section 4 writes these from the raw binary.")
for msg_dir in written:
    try:
        sample = pl.scan_parquet(msg_dir / "*.parquet").head(5).collect()
        name = MESSAGE_SPECS.get(msg_dir.name, {}).get("name", "Unknown")
        print(f"  [OK] {msg_dir.name} ({name:25}): cols={list(sample.columns)[:4]}...")
    except Exception as e:
        print(f"  [FAIL] {msg_dir.name}: {e}")

Schema Compatibility Check:
--------------------------------------------------
  [OK] A (Add Order                ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] C (Order Executed with Price): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] D (Order Delete             ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] E (Order Executed           ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] F (Add Order MPID           ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] H (Trading Action           ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']...
  [OK] I (NOII                     ): cols=['stock_locate', 'tracking_number', 'timestamp', 'paired_shares']...
  [OK] J (LULD Auction Collar      ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']

  [OK] L (Market Participant Position): cols=['stock_locate', 'tracking_number', 'timestamp', 'mpid']...
  [OK] P (Trade                    ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] Q (Cross Trade              ): cols=['stock_locate', 'tracking_number', 'timestamp', 'shares']...
  [OK] R (Stock Directory          ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']...
  [OK] S (System Event             ): cols=['stock_locate', 'tracking_number', 'timestamp', 'event_code']...


  [OK] U (Order Replace            ): cols=['stock_locate', 'tracking_number', 'timestamp', 'original_order_reference_number']...
  [OK] V (MWCB Decline Level       ): cols=['stock_locate', 'tracking_number', 'timestamp', 'level_1']...
  [OK] X (Order Cancel             ): cols=['stock_locate', 'tracking_number', 'timestamp', 'order_reference_number']...
  [OK] Y (Reg SHO Restriction      ): cols=['stock_locate', 'tracking_number', 'timestamp', 'stock']...


## 6. 使用Rust进行生产解析

上面的Python解析器每次循环迭代解码一条消息，一个交易日包含几亿条这样的消息。相同的协议在Rust中解析通过内存映射读取文件，并在不先复制的情况下解包每条消息，因此它既不支付每条消息的解释器开销，也不在内存中保留解码后的消息。

**仓库**：[github.com/ml4t/itch-parser](https://github.com/ml4t/itch-parser)

书中表3.1中的数据，对于同一台机器上的13GB文件，Python大约需要二十三分钟和大约8GB的内存，而Rust则不到五分钟和不到500MB。墙钟时间随着磁盘和CPU的变化而变化，所以将其视为数量级而不是比率：差距足够大，足以决定你选择哪个解析器，而且不够稳定，不能精确到小数位。这个笔记本上面打印的吞吐量是相同比较的Python部分，在运行它的机器上测量的。

### 安装

bash
# 克隆仓库
git clone https://github.com/ml4t/itch-parser.git
cd itch-parser

# 构建发布二进制文件
cargo build --release


### 使用

bash
# 解析ITCH文件（适用于.gz或未压缩的文件）
./target/release/itch_parser <input_file> <output_dir> <MMDDYYYY>

# 示例
./target/release/itch_parser data/01302020.NASDAQ_ITCH50.gz ./messages 01302020


输出是相同类型的Parquet文件，按消息类型分区，与这个笔记本中的Python代码和下游分析兼容。

### 何时使用哪个

| 用例 | 推荐 |
|------|------|
| 学习协议 | Python（这个笔记本） |
| 调试解析问题 | Python |
| 处理单日数据 | 任意 |
| 多日回测 | **Rust** |
| 生产流程 | **Rust** |

两个解析器都写入相同的Parquet模式，所以无论是哪种方式解析，解析后的数据都可以无缝地喂养后续的每个笔记本。

## 关键要点

1. **协议按订单消息。** 每条改变订单簿的消息都会命名它所作用的单个订单，并标注到纳秒级别，这是实现订单簿重建的基础。会话级消息 - `S` 代表市场事件，`R` 代表股票目录，`Q` 代表拍卖交叉 - 不包含订单引用，而是描述场所而不是单个订单。
2. **七种消息类型携带订单簿。** `A` 和 `F` 添加订单，`E` 和 `C` 执行订单，`X` 取消部分订单，`D` 删除订单，`U` 替换订单。
3. **数字以编码形式到达。** 价格是整数，有四个隐含的小数位，时间戳是午夜以来的纳秒数，因此在使用之前都需要转换。
4. **分批解析，而不是一次性解析。** 通过消息类型进行缓冲，并将数据刷新到 Parquet，这是确保一天的数据不会因为不适合内存而需要。
5. **解析器取决于你需要多少天。** Python 读取清晰，每天大约需要二十分钟；Rust 在极短的时间内产生相同的模式。

### 已知限制

- 解析只覆盖一个场所。NASDAQ ITCH 只看到 NASDAQ 路由的活动，而不是合并的带子，因此这里的计数是一个场所的符号交易份额，而不是全部。
- `FMT_DICT` 之外的消息类型被计数并跳过，而不是解码。
- 上文中引用的 Rust 时间是在另一台机器上测量的；这个笔记本只测量其自身的 Python 解析。

### 下一步

- **订单簿重建**：`02_itch_lob_reconstruction`
- **交易活动概述**：`05_itch_trading_activity`（包括 E/C 丰富化）

---

## 参考

Bouchaud, J.-P., Bonart, J., Donier, J., & Gould, M. (2018).
*Trades, Quotes and Prices: Financial Markets Under the Microscope*.
Cambridge University Press.
[https://doi.org/10.1017/9781009028943](https://doi.org/10.1017/9781009028943)